# Crawl Conservative Pathfinder Agent

This notebook writes a rule-based agent for the Kaggle Maze Crawler competition.

Strategy notes:
- Keep the factory alive first; never walk off the north edge.
- Build one worker as a wall opener, then conserve factory energy instead of scout-spamming.
- Use BFS over discovered walls to keep moving north when direct movement is blocked.
- Avoid emergency jumps into sealed cells; prefer normal movement before jumping.
- Use safe reservations to reduce friendly collisions, and let the worker detour to visible crystals.


In [ ]:
%%writefile submission.py
from collections import deque


FACTORY, SCOUT, WORKER, MINER = 0, 1, 2, 3
DIRS = ("NORTH", "EAST", "WEST", "SOUTH")
OFFSETS = {
    "NORTH": (0, 1),
    "SOUTH": (0, -1),
    "EAST": (1, 0),
    "WEST": (-1, 0),
}
WALL_BITS = {"NORTH": 1, "EAST": 2, "SOUTH": 4, "WEST": 8}
MAX_ENERGY = {SCOUT: 100, WORKER: 300, MINER: 500}


def agent(obs, config):
    actions = {}
    width = config.width
    south = obs.southBound
    north = obs.northBound

    my_robots = {
        uid: data for uid, data in obs.robots.items()
        if data[4] == obs.player
    }
    occupied = {(data[1], data[2]): uid for uid, data in my_robots.items()}
    reserved = set()
    selected = {}

    crystals = {}
    for key, value in obs.crystals.items():
        col, row = map(int, key.split(","))
        if south <= row <= north:
            crystals[(col, row)] = value

    def wall_at(col, row):
        idx = (row - south) * width + col
        if 0 <= col < width and 0 <= idx < len(obs.walls) and obs.walls[idx] != -1:
            return obs.walls[idx]
        return 0

    def can_move(col, row, direction):
        dc, dr = OFFSETS[direction]
        next_col, next_row = col + dc, row + dr
        if not (0 <= next_col < width and south <= next_row <= north):
            return False
        return not (wall_at(col, row) & WALL_BITS[direction])

    def can_jump_to_open_cell(col, row, direction):
        dc, dr = OFFSETS[direction]
        target = (col + 2 * dc, row + 2 * dr)
        target_col, target_row = target
        if not (0 <= target_col < width and south <= target_row <= north):
            return False
        if target in reserved or target in occupied:
            return False
        return any(can_move(target_col, target_row, exit_dir) for exit_dir in DIRS)

    def safe_dirs(col, row, prefs, uid):
        result = []
        for direction in prefs:
            if not can_move(col, row, direction):
                continue
            dc, dr = OFFSETS[direction]
            target = (col + dc, row + dr)
            if target in reserved:
                continue
            occupant = occupied.get(target)
            if occupant is not None and occupant != uid:
                if occupant not in selected or selected.get(occupant) == "IDLE":
                    continue
            result.append(direction)
        return result

    def bfs_first_step(start, goals, uid, max_depth=8):
        goals = set(goals)
        if not goals:
            return None
        queue = deque([(start, None, 0)])
        seen = {start}
        while queue:
            (col, row), first, dist = queue.popleft()
            if dist > 0 and (col, row) in goals:
                return first
            if dist >= max_depth:
                continue
            for direction in DIRS:
                if not can_move(col, row, direction):
                    continue
                dc, dr = OFFSETS[direction]
                nxt = (col + dc, row + dr)
                if nxt in seen:
                    continue
                if nxt != start and nxt in occupied:
                    continue
                seen.add(nxt)
                queue.append((nxt, first or direction, dist + 1))
        return None

    def choose_move(uid, data, prefs):
        col, row = data[1], data[2]
        options = safe_dirs(col, row, prefs, uid)
        return options[0] if options else "IDLE"

    def record_action(uid, action, extra_reserve=None):
        selected[uid] = action
        actions[uid] = action
        col, row = my_robots[uid][1], my_robots[uid][2]
        if action in OFFSETS:
            dc, dr = OFFSETS[action]
            reserved.add((col + dc, row + dr))
        elif action.startswith("JUMP_"):
            direction = action.split("_")[1]
            dc, dr = OFFSETS[direction]
            reserved.add((col + 2 * dc, row + 2 * dr))
        else:
            reserved.add((col, row))
        if extra_reserve is not None:
            reserved.add(extra_reserve)

    counts = {
        robot_type: sum(1 for data in my_robots.values() if data[0] == robot_type)
        for robot_type in (FACTORY, SCOUT, WORKER, MINER)
    }
    ordered = sorted(
        my_robots.items(),
        key=lambda item: (
            0 if item[1][0] == FACTORY else
            1 if item[1][0] == WORKER else
            2 if item[1][0] == MINER else
            3,
            item[0],
        ),
    )

    for uid, data in ordered:
        robot_type, col, row, energy = data[0], data[1], data[2], data[3]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0
        walls = wall_at(col, row)
        gap = row - south

        if robot_type == FACTORY:
            emergency = south > 0 and gap <= 2
            can_jump_north = (
                move_cd <= 1
                and jump_cd <= 1
                and can_jump_to_open_cell(col, row, "NORTH")
            )

            if (walls & WALL_BITS["NORTH"]) and can_jump_north:
                record_action(uid, "JUMP_NORTH")
                continue

            if move_cd <= 1:
                target_row = min(north, row + 1)
                targets = [(target_col, target_row) for target_col in range(width)]
                step = bfs_first_step((col, row), targets, uid, max_depth=6)
                if step and step in safe_dirs(col, row, [step], uid):
                    record_action(uid, step)
                    continue
                side_a, side_b = ("EAST", "WEST") if col < width // 2 else ("WEST", "EAST")
                prefs = ["NORTH", side_a, side_b] if emergency else ["NORTH", side_a, side_b, "SOUTH"]
                fallback = choose_move(uid, data, prefs)
                if fallback != "IDLE":
                    record_action(uid, fallback)
                    continue
                if emergency and can_jump_north:
                    record_action(uid, "JUMP_NORTH")
                    continue

            spawn = (col, row + 1)
            if (
                build_cd <= 1
                and gap >= 2
                and row + 1 <= north
                and can_move(col, row, "NORTH")
                and spawn not in occupied
                and spawn not in reserved
                and counts[WORKER] < 1
                and energy >= config.workerCost + 200
            ):
                record_action(uid, "BUILD_WORKER", spawn)
                counts[WORKER] += 1
                continue

            record_action(uid, "IDLE")
            continue

        if robot_type == WORKER and (walls & WALL_BITS["NORTH"]):
            if energy >= config.wallRemoveCost + 20 and row + 1 <= north:
                record_action(uid, "REMOVE_NORTH")
                continue

        if move_cd > 1:
            record_action(uid, "IDLE")
            continue

        goals = []
        max_energy = MAX_ENERGY.get(robot_type, 10**9)
        if max_energy - energy > 3:
            goals = [
                pos for pos, value in sorted(
                    crystals.items(),
                    key=lambda item: (
                        abs(item[0][0] - col) + abs(item[0][1] - row) - item[1] / 25,
                        -item[1],
                    ),
                )
            ]

        step = bfs_first_step((col, row), goals, uid, max_depth=8) if goals else None
        if step and step in safe_dirs(col, row, [step], uid):
            record_action(uid, step)
            continue

        side_a, side_b = ("EAST", "WEST") if col < width // 2 else ("WEST", "EAST")
        record_action(uid, choose_move(uid, data, ["NORTH", side_a, side_b, "SOUTH"]))

    return actions
